In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BASE_CV_DIR = "../data/train/cv_folds_aug"
RESULTS_DIR = "cv_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
CSV_RESULTS_PATH = os.path.join(RESULTS_DIR, "cross_validation_metrics.csv")

/home/ugnius/Documents/KTU/IV-kursas/II-semestras/BBP/V2/model_tests/.venv/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>
  data = fetch_version_info()


Using device: cuda


In [ ]:
class FilthDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        
        self.image_names = []
        if os.path.exists(images_dir):
            self.image_names = sorted([f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg', '.jpeg', '.JPG'))])
        else:
            print(f"Warning: Images directory {images_dir} does not exist.")

        self.transform = A.Compose([
            A.Resize(512, 512),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_names)
        
    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        name_lower = img_name.lower()
        if "meat" in name_lower:
            category = "meat"
        elif "vegetables" in name_lower:
            category = "vegetables"
        else:
            category = "clean"
            
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask_path = os.path.join(self.masks_dir, img_name)
        if not os.path.exists(mask_path):
            mask_path = os.path.splitext(mask_path)[0] + '.png'
            
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path)
            mask = np.any(mask > 0, axis=-1).astype(np.float32)
        else:
            mask = np.zeros(image.shape[:2], dtype=np.float32)

        augmented = self.transform(image=image, mask=mask)
        image = augmented['image']
        mask = augmented['mask'].unsqueeze(0) 

        return image, mask, category, img_name

In [ ]:
def train_and_validate(model, train_loader, val_loader, loss_fn, threshold, device, weights_save_path, epochs=50, patience=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    best_miou = -1.0
    patience_counter = 0

    for epoch in range(epochs):
        print(f"\n  Epoch {epoch+1:02d}/{epochs}")
        
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc="    Training", leave=False)
        
        for images, masks, _, _ in train_bar:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            train_bar.set_postfix(batch_loss=f"{loss.item():.4f}")
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_tp, val_fp, val_fn, val_tn = 0, 0, 0, 0
        val_bar = tqdm(val_loader, desc="    Validating", leave=False)
        
        with torch.no_grad():
            for images, masks, _, _ in val_bar:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                preds = (torch.sigmoid(outputs) > threshold).float()
                
                tp, fp, fn, tn = smp.metrics.get_stats(preds.long(), masks.long(), mode='binary')
                val_tp += tp.sum().item()
                val_fp += fp.sum().item()
                val_fn += fn.sum().item()
                val_tn += tn.sum().item()

        val_miou = val_tp / (val_tp + val_fp + val_fn + 1e-7)
        
        print(f"    -> Train Loss: {train_loss:.4f} | Val mIoU: {val_miou:.4f} | Patience: {patience_counter}/{patience}")

        if val_miou > best_miou:
            best_miou = val_miou
            patience_counter = 0
            torch.save(model.state_dict(), weights_save_path)
            print(f"    -> [Saved new best weights based on mIoU]")
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"    -> Early stopping triggered at epoch {epoch+1}!")
            break

    if os.path.exists(weights_save_path):
        model.load_state_dict(torch.load(weights_save_path, map_location=device))
        
    return model

def evaluate_fold(model, val_loader, threshold, device):
    model.eval()
    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
    eval_bar = tqdm(val_loader, desc="    Final Eval", leave=False)
    
    with torch.no_grad():
        for images, masks, _, _ in eval_bar:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > threshold).float()
            
            tp, fp, fn, tn = smp.metrics.get_stats(preds.long(), masks.long(), mode='binary')
            total_tp += tp.sum().item()
            total_fp += fp.sum().item()
            total_fn += fn.sum().item()
            total_tn += tn.sum().item()

    accuracy = (total_tp + total_tn) / (total_tp + total_fp + total_fn + total_tn + 1e-7)
    precision = total_tp / (total_tp + total_fp + 1e-7)
    recall = total_tp / (total_tp + total_fn + 1e-7)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
    iou = total_tp / (total_tp + total_fp + total_fn + 1e-7)

    return {
        "Accuracy": accuracy, "mIoU": iou, "Precision": precision, 
        "Recall": recall, "F1": f1,
        "TP_Pixels": total_tp, "FP_Pixels": total_fp, 
        "FN_Pixels": total_fn, "TN_Pixels": total_tn
    }

In [ ]:

def visualize_val_set(model, val_loader, threshold, device, save_dir):
    """Saves visual comparisons for the entire validation fold."""
    os.makedirs(save_dir, exist_ok=True)
    model.eval()

    inv_normalize = A.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225],
        max_pixel_value=1.0
    )

    with torch.no_grad():
        for images, masks, _, img_names in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > threshold).float()
            
            for i in range(images.size(0)):
                img_np = images[i].cpu().permute(1, 2, 0).numpy()
                img_denorm = inv_normalize(image=img_np)['image']
                img_denorm = (np.clip(img_denorm, 0, 1) * 255).astype(np.uint8)
                
                gt_mask = masks[i].cpu().squeeze().numpy()
                pred_mask = preds[i].cpu().squeeze().numpy()
                
                gt_overlay = img_denorm.copy()
                gt_overlay[gt_mask == 1] = gt_overlay[gt_mask == 1] * 0.5 + np.array([255, 0, 0]) * 0.5
                
                pred_overlay = img_denorm.copy()
                pred_overlay[pred_mask == 1] = pred_overlay[pred_mask == 1] * 0.5 + np.array([0, 0, 255]) * 0.5
                
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                axes[0].imshow(img_denorm); axes[0].set_title(f"Base: {img_names[i]}"); axes[0].axis('off')
                axes[1].imshow(gt_overlay); axes[1].set_title("Ground Truth (Red)"); axes[1].axis('off')
                axes[2].imshow(pred_overlay); axes[2].set_title("Prediction (Blue)"); axes[2].axis('off')
                
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, f"{img_names[i]}_result.png"))
                plt.close(fig)

def save_intermediate_metrics(results_list):
    """Saves to CSV and prints current progress including normalized confusion matrix."""
    df = pd.DataFrame(results_list)
    df.to_csv(CSV_RESULTS_PATH, index=False)
    
    latest = results_list[-1]
    
    tp, fp = int(latest['TP_Pixels']), int(latest['FP_Pixels'])
    fn, tn = int(latest['FN_Pixels']), int(latest['TN_Pixels'])
    
    pos_total = tp + fn + 1e-7
    neg_total = fp + tn + 1e-7
    
    tp_norm, fn_norm = (tp / pos_total) * 100, (fn / pos_total) * 100
    fp_norm, tn_norm = (fp / neg_total) * 100, (tn / neg_total) * 100
    
    print(f"\n  >>> FOLD COMPLETE: {latest['Model']} | {latest['Loss']} | Fold {latest['Fold']}")
    print(f"      - mIoU (Patience Metric): {latest['mIoU']:.4f}")
    print(f"      - F1 / Recall / Prec:    {latest['F1']:.4f} / {latest['Recall']:.4f} / {latest['Precision']:.4f}")
    
    print(f"\n      [Absolute Pixel Matrix]")
    print(f"                       Pred Filth      Pred Clean")
    print(f"        Actual Filth | TP: {tp:<10}  FN: {fn:<10}")
    print(f"        Actual Clean | FP: {fp:<10}  TN: {tn:<10}")
    
    print(f"\n      [Row-Normalized Matrix (%)]")
    print(f"                       Pred Filth      Pred Clean")
    print(f"        Actual Filth | TP: {tp_norm:>5.2f}%       FN: {fn_norm:>5.2f}%")
    print(f"        Actual Clean | FP: {fp_norm:>5.2f}%       TN: {tn_norm:>5.2f}%")
    
    print(f"\n  [Progress saved to {CSV_RESULTS_PATH}]\n")

In [ ]:
def get_model(model_name):
    if model_name == "DeepLabV3Plus":
        return smp.DeepLabV3Plus(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
    elif model_name == "UnetPlusPlus":
        return smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
    elif model_name == "Segformer":
        return smp.Segformer(encoder_name="mit_b0", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
    else:
        raise ValueError(f"Unknown model: {model_name}")

def get_loss_config(loss_name):
    if loss_name == "BCE":
        return torch.nn.BCEWithLogitsLoss(), 0.5 
    elif loss_name == "Tversky":
        return smp.losses.TverskyLoss(mode='binary', alpha=0.2, beta=0.8), 0.5
    else:
        raise ValueError(f"Unknown loss: {loss_name}")

def append_to_csv(result_dict, csv_path):
    """Safely appends a new result row to the CSV on disk."""
    df_new = pd.DataFrame([result_dict])
    if os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.to_csv(csv_path, index=False)
    else:
        df_new.to_csv(csv_path, index=False)

def run_cross_validation(model_name, loss_name, num_folds=5):
    """Executes the CV pipeline for a specific model AND specific loss function."""
    print(f"\n{'='*70}")
    print(f"EXPERIMENT START: Model = {model_name} | Loss = {loss_name}")
    print(f"{'='*70}")
    
    fold_results = []
    
    for fold in range(1, num_folds + 1):
        print(f"\n--- Processing FOLD {fold} ---")
        
        train_img_dir = os.path.join(BASE_CV_DIR, f"fold_{fold}", "train", "images")
        train_mask_dir = os.path.join(BASE_CV_DIR, f"fold_{fold}", "train", "masks")
        val_img_dir = os.path.join(BASE_CV_DIR, f"fold_{fold}", "val", "images")
        val_mask_dir = os.path.join(BASE_CV_DIR, f"fold_{fold}", "val", "masks")
        
        train_dataset = FilthDataset(train_img_dir, train_mask_dir)
        val_dataset = FilthDataset(val_img_dir, val_mask_dir)
        
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
        
        model = get_model(model_name)
        loss_fn, threshold = get_loss_config(loss_name)
        
        weights_path = os.path.join(RESULTS_DIR, f"{model_name}_{loss_name}_fold{fold}_best.pth")
        model = train_and_validate(
            model=model, train_loader=train_loader, val_loader=val_loader, 
            loss_fn=loss_fn, threshold=threshold, device=device, 
            weights_save_path=weights_path, epochs=50, patience=5
        )
        
        metrics = evaluate_fold(model, val_loader, threshold, device)
        
        result_entry = {"Model": model_name, "Loss": loss_name, "Fold": fold, **metrics}
        
        append_to_csv(result_entry, CSV_RESULTS_PATH)
        save_intermediate_metrics([result_entry]) 
        
        vis_dir = os.path.join(RESULTS_DIR, "visuals", f"{model_name}_{loss_name}_fold{fold}")
        print(f"  [Generating visual overlays in: {vis_dir}]\n")
        visualize_val_set(model, val_loader, threshold, device, vis_dir)
        
        fold_results.append(metrics)
        del model
        torch.cuda.empty_cache()

    print(f"\n✅ All {num_folds} folds for {model_name} with {loss_name} completed.")
    
    print(f"\n{'='*70}")
    print(f"AGGREGATE SUMMARY: {model_name} + {loss_name} (Over {num_folds} Folds)")
    print(f"{'='*70}")
    
    avg_miou = np.mean([f['mIoU'] for f in fold_results])
    avg_f1 = np.mean([f['F1'] for f in fold_results])
    avg_recall = np.mean([f['Recall'] for f in fold_results])
    avg_precision = np.mean([f['Precision'] for f in fold_results])
    
    total_tp = int(sum([f['TP_Pixels'] for f in fold_results]))
    total_fp = int(sum([f['FP_Pixels'] for f in fold_results]))
    total_fn = int(sum([f['FN_Pixels'] for f in fold_results]))
    total_tn = int(sum([f['TN_Pixels'] for f in fold_results]))
    
    pos_total = total_tp + total_fn + 1e-7
    neg_total = total_fp + total_tn + 1e-7
    
    tp_norm, fn_norm = (total_tp / pos_total) * 100, (total_fn / pos_total) * 100
    fp_norm, tn_norm = (total_fp / neg_total) * 100, (total_tn / neg_total) * 100
    
    print(f"    Average mIoU:      {avg_miou:.4f}")
    print(f"    Average F1 Score:  {avg_f1:.4f}")
    print(f"    Average Recall:    {avg_recall:.4f}")
    print(f"    Average Precision: {avg_precision:.4f}")
    
    print(f"\n    [Aggregate Absolute Pixel Matrix]")
    print(f"                       Pred Filth      Pred Clean")
    print(f"        Actual Filth | TP: {total_tp:<10}  FN: {total_fn:<10}")
    print(f"        Actual Clean | FP: {total_fp:<10}  TN: {total_tn:<10}")
    
    print(f"\n    [Aggregate Row-Normalized Matrix (%)]")
    print(f"                       Pred Filth      Pred Clean")
    print(f"        Actual Filth | TP: {tp_norm:>5.2f}%       FN: {fn_norm:>5.2f}%")
    print(f"        Actual Clean | FP: {fp_norm:>5.2f}%       TN: {tn_norm:>5.2f}%\n")

In [6]:
run_cross_validation(model_name="UnetPlusPlus", loss_name="BCE")


EXPERIMENT START: Model = UnetPlusPlus | Loss = BCE

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.2976 | Val mIoU: 0.5113 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.0642 | Val mIoU: 0.7124 | Patience: 4/5
    -> Early stopping triggered at epoch 13!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: UnetPlusPlus | BCE | Fold 5
      - mIoU (Patience Metric): 0.7206
      - F1 / Recall / Prec:    0.8376 / 0.8194 / 0.8566

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1112898     FN: 245333    
        Actual Clean | FP: 186277      TN: 14184132  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 81.94%       FN: 18.06%
        Actual Clean | FP:  1.30%       TN: 98.70%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/UnetPlusPlus_BCE_fold5]


✅ All 5 folds for UnetPlusPlus with BCE completed.

AGGREGATE SUMMARY: UnetPlusPlus + BCE (Over 5 Folds)
    Average mIoU:      0.6952
    Average F1 Score:  0.8199
    Average Recall:    0.7947
    Average Precision: 0.8478

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth |

In [7]:
run_cross_validation(model_name="UnetPlusPlus", loss_name="Tversky")


EXPERIMENT START: Model = UnetPlusPlus | Loss = Tversky

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.3252 | Val mIoU: 0.3924 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.1305 | Val mIoU: 0.6686 | Patience: 4/5
    -> Early stopping triggered at epoch 9!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: UnetPlusPlus | Tversky | Fold 5
      - mIoU (Patience Metric): 0.7016
      - F1 / Recall / Prec:    0.8247 / 0.8519 / 0.7991

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1157088     FN: 201143    
        Actual Clean | FP: 290881      TN: 14079528  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 85.19%       FN: 14.81%
        Actual Clean | FP:  2.02%       TN: 97.98%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/UnetPlusPlus_Tversky_fold5]


✅ All 5 folds for UnetPlusPlus with Tversky completed.

AGGREGATE SUMMARY: UnetPlusPlus + Tversky (Over 5 Folds)
    Average mIoU:      0.6753
    Average F1 Score:  0.8059
    Average Recall:    0.8518
    Average Precision: 0.7659

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
      

In [8]:
run_cross_validation(model_name="Segformer", loss_name="BCE")


EXPERIMENT START: Model = Segformer | Loss = BCE

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.2499 | Val mIoU: 0.5591 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.0781 | Val mIoU: 0.7207 | Patience: 4/5
    -> Early stopping triggered at epoch 18!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: Segformer | BCE | Fold 5
      - mIoU (Patience Metric): 0.7254
      - F1 / Recall / Prec:    0.8409 / 0.8074 / 0.8772

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1096609     FN: 261622    
        Actual Clean | FP: 153467      TN: 14216942  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 80.74%       FN: 19.26%
        Actual Clean | FP:  1.07%       TN: 98.93%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/Segformer_BCE_fold5]


✅ All 5 folds for Segformer with BCE completed.

AGGREGATE SUMMARY: Segformer + BCE (Over 5 Folds)
    Average mIoU:      0.7075
    Average F1 Score:  0.8284
    Average Recall:    0.8191
    Average Precision: 0.8385

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 5351338

In [9]:
run_cross_validation(model_name="Segformer", loss_name="Tversky")


EXPERIMENT START: Model = Segformer | Loss = Tversky

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.2502 | Val mIoU: 0.5207 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.1187 | Val mIoU: 0.6857 | Patience: 4/5
    -> Early stopping triggered at epoch 16!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: Segformer | Tversky | Fold 5
      - mIoU (Patience Metric): 0.6874
      - F1 / Recall / Prec:    0.8147 / 0.8870 / 0.7534

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1204694     FN: 153537    
        Actual Clean | FP: 394330      TN: 13976079  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 88.70%       FN: 11.30%
        Actual Clean | FP:  2.74%       TN: 97.26%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/Segformer_Tversky_fold5]


✅ All 5 folds for Segformer with Tversky completed.

AGGREGATE SUMMARY: Segformer + Tversky (Over 5 Folds)
    Average mIoU:      0.6692
    Average F1 Score:  0.8013
    Average Recall:    0.8682
    Average Precision: 0.7452

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Fil

In [10]:
run_cross_validation(model_name="DeepLabV3Plus", loss_name="BCE")


EXPERIMENT START: Model = DeepLabV3Plus | Loss = BCE

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.2852 | Val mIoU: 0.5617 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.0821 | Val mIoU: 0.7176 | Patience: 4/5
    -> Early stopping triggered at epoch 21!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: DeepLabV3Plus | BCE | Fold 5
      - mIoU (Patience Metric): 0.7241
      - F1 / Recall / Prec:    0.8400 / 0.8332 / 0.8468

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1131683     FN: 226548    
        Actual Clean | FP: 204698      TN: 14165711  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 83.32%       FN: 16.68%
        Actual Clean | FP:  1.42%       TN: 98.58%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/DeepLabV3Plus_BCE_fold5]


✅ All 5 folds for DeepLabV3Plus with BCE completed.

AGGREGATE SUMMARY: DeepLabV3Plus + BCE (Over 5 Folds)
    Average mIoU:      0.6905
    Average F1 Score:  0.8166
    Average Recall:    0.7956
    Average Precision: 0.8391

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Fil

In [11]:
run_cross_validation(model_name="DeepLabV3Plus", loss_name="Tversky")


EXPERIMENT START: Model = DeepLabV3Plus | Loss = Tversky

--- Processing FOLD 1 ---

  Epoch 01/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.3080 | Val mIoU: 0.5182 | Patience: 0/5
    -> [Saved new best weights based on mIoU]

  Epoch 02/50


    Training:   0%|          | 0/640 [00:00<?, ?it/s]

...

    Training:   0%|          | 0/640 [00:00<?, ?it/s]

    Validating:   0%|          | 0/60 [00:00<?, ?it/s]

    -> Train Loss: 0.1244 | Val mIoU: 0.6597 | Patience: 4/5
    -> Early stopping triggered at epoch 13!


    Final Eval:   0%|          | 0/60 [00:00<?, ?it/s]


  >>> FOLD COMPLETE: DeepLabV3Plus | Tversky | Fold 5
      - mIoU (Patience Metric): 0.6802
      - F1 / Recall / Prec:    0.8097 / 0.8682 / 0.7586

      [Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 1179224     FN: 179007    
        Actual Clean | FP: 375335      TN: 13995074  

      [Row-Normalized Matrix (%)]
                       Pred Filth      Pred Clean
        Actual Filth | TP: 86.82%       FN: 13.18%
        Actual Clean | FP:  2.61%       TN: 97.39%

  [Progress saved to cv_results/cross_validation_metrics.csv]

  [Generating visual overlays in: cv_results/visuals/DeepLabV3Plus_Tversky_fold5]


✅ All 5 folds for DeepLabV3Plus with Tversky completed.

AGGREGATE SUMMARY: DeepLabV3Plus + Tversky (Over 5 Folds)
    Average mIoU:      0.6558
    Average F1 Score:  0.7917
    Average Recall:    0.8344
    Average Precision: 0.7546

    [Aggregate Absolute Pixel Matrix]
                       Pred Filth      Pred Clean
  